# Mangaka : entraînement sur GPU Kaggle

Ce notebook clone le dépôt, télécharge le dataset AnimeGAN, puis lance les scripts du dépôt
(rien n'est réécrit ici, tout le code vient de `src/`).

**Avant de lancer** : *Settings* → *Accelerator* : **GPU T4 x2** (ou P100), et *Internet* : **On**.
Remplir `REPO_URL` (et `BRANCH`) dans la cellule suivante.

À la fin, télécharger `/kaggle/working/resultats.zip` (checkpoints, ONNX, metrics.json, base MLflow)
et le décompresser à la racine du dépôt sur le PC, puis faire `dvc add` (voir README).

In [ ]:
REPO_URL = "https://github.com/<compte>/<depot>.git"  # si pas de dataset Kaggle du code
BRANCH = "develop"
STYLES = ["Hayao", "Shinkai"]  # modèle avancé : un générateur par style
EPOCHS_ANIMEGAN = 30
EPOCHS_CYCLEGAN = 20
WORKERS = 4
# Reprise après un run coupé par la limite de 12 h de Kaggle : ajouter la sortie du run
# précédent en entrée (« Add Input » > Notebook). Les modèles déjà évalués sont sautés,
# un générateur AnimeGAN interrompu repart de son dernier checkpoint (sauvé à chaque époque).
EPOCHS_DONE = {"Hayao": 11}        # époques déjà faites par style dans le run précédent
EPOCHS_TARGET = {"Hayao": 20, "Shinkai": 10}  # total visé par style pour ce run de reprise


In [ ]:
!nvidia-smi -L
# Code : soit un dataset Kaggle privé contenant le dépôt (ajouté via « Add Input »),
# soit un clone Git de REPO_URL.
import glob, os, shutil
# (on ignore la copie du dépôt présente dans la sortie d'un run précédent)
found = [p for p in glob.glob("/kaggle/input/**/src/models/train.py", recursive=True) if "/repo/" not in p]
if found:
    shutil.copytree(os.path.dirname(os.path.dirname(os.path.dirname(found[0]))), "/kaggle/working/repo", dirs_exist_ok=True)
else:
    !git clone --branch $BRANCH $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements-train.txt


## Données : dataset officiel AnimeGAN (release GitHub, 793 Mo)

In [ ]:
!wget -q https://github.com/TachibanaYoshino/AnimeGAN/releases/download/dataset-1/dataset.zip -O /kaggle/working/dataset.zip
!mkdir -p data/raw && unzip -q /kaggle/working/dataset.zip -d /kaggle/working/dataset_unzip
!ls /kaggle/working/dataset_unzip/*

In [ ]:
# Recopie vers l'arborescence attendue par src/data/dataset.py (voir data/README.md),
# puis contrôle qu'aucune photo de test n'est aussi dans les photos d'entraînement.
!python -m src.data.prepare_raw --src /kaggle/working/dataset_unzip --dst data/raw

In [ ]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////kaggle/working/repo/mlflow.db"
for style in STYLES:
    !python -m src.data.dataset --style {style}

## Reprise éventuelle d'un run précédent


In [ ]:
# Récupère checkpoints/, models/ et mlflow.db d'un run précédent s'il est en entrée.
import glob, os, shutil
prev = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/checkpoints", recursive=True)]
RESUMING = bool(prev)
if RESUMING:
    src = prev[0]
    print("Reprise depuis", src)
    for name in ("checkpoints", "models"):
        if os.path.isdir(os.path.join(src, name)):
            shutil.copytree(os.path.join(src, name), name, dirs_exist_ok=True)
    if os.path.isfile(os.path.join(src, "mlflow.db")):
        shutil.copy(os.path.join(src, "mlflow.db"), "mlflow.db")
!ls -R checkpoints models 2>/dev/null | head -30


## Baseline CycleGAN (128×128, Hayao)

In [ ]:
if os.path.isfile("models/cyclegan-hayao.onnx"):
    print("Baseline CycleGAN déjà entraînée et évaluée : sautée")
else:
    !python -m src.models.train_cyclegan --style Hayao --epochs $EPOCHS_CYCLEGAN --workers $WORKERS
    !python -m src.models.evaluate --arch cyclegan --size 128 --style Hayao --checkpoint checkpoints/cyclegan-hayao/generator.pt


## Modèle avancé AnimeGAN (256×256), un générateur par style

Shinkai repart des poids Hayao (`--resume`) pour économiser du temps GPU.
En reprise, chaque style ne fait que les époques qui manquent pour atteindre `EPOCHS_TARGET`.


In [ ]:
def animegan(style, init_from=None):
    ckpt = f"checkpoints/{style.lower()}/generator.pt"
    done = EPOCHS_DONE.get(style, 0) if RESUMING and os.path.isfile(ckpt) else 0
    total = EPOCHS_TARGET.get(style, EPOCHS_ANIMEGAN) if RESUMING else EPOCHS_ANIMEGAN
    if done:
        resume, init, epochs = ckpt, 0, total - done
    elif init_from:
        resume, init, epochs = init_from, 0, total
    else:
        resume, init, epochs = "", 2, total
    print(f"{style} : {epochs} époque(s), reprise={resume or 'non'}")
    if epochs > 0:
        args = f"--resume {resume}" if resume else ""
        !python -m src.models.train --style {style} {args} --init-epochs {init} --epochs {epochs} --workers $WORKERS
    !python -m src.models.evaluate --style {style} --checkpoint {ckpt} --register

animegan("Hayao")


In [ ]:
if "Shinkai" in STYLES:
    animegan("Shinkai", init_from="checkpoints/hayao/generator.pt")


In [ ]:
!cat models/metrics.json
!cd /kaggle/working/repo && zip -qr /kaggle/working/resultats.zip checkpoints models mlflow.db mlruns
!ls -lh /kaggle/working/resultats.zip